# 🍎 Fruit & Vegetable Dataset Exploration (EDA)

This notebook explores the **Fruits and Vegetables Image Recognition Dataset**:
- 36 distinct produce classes
- Class distribution across `train`, `validation`, and `test` splits
- Visual gallery displaying sample images from each produce category
- Image dimensions, channels, and aspect ratio statistics

In [ ]:
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Load data paths from project configuration
with open('../train_config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

train_dir = Path(cfg['data']['train_dir'])
val_dir   = Path(cfg['data']['val_dir'])
test_dir  = Path(cfg['data']['test_dir'])

print(f"Train directory: {train_dir}")
print(f"Validation dir : {val_dir}")
print(f"Test directory : {test_dir}")

## 1. Class Distribution Analysis
Let's count how many images exist per category in each split.

In [ ]:
def count_split(split_dir):
    counts = {}
    for class_folder in sorted([d for d in split_dir.iterdir() if d.is_dir()]):
        n = len(list(class_folder.glob('*.*')))
        counts[class_folder.name] = n
    return counts

train_counts = count_split(train_dir)
val_counts   = count_split(val_dir)
test_counts  = count_split(test_dir)

df_counts = pd.DataFrame({
    'Train': train_counts,
    'Validation': val_counts,
    'Test': test_counts,
})
df_counts['Total'] = df_counts.sum(axis=1)
display(df_counts.sort_values(by='Total', ascending=False))

In [ ]:
# Plot class distribution
plt.figure(figsize=(16, 6))
df_counts['Train'].sort_values(ascending=False).plot(kind='bar', color='#2ecc71', edgecolor='#27ae60')
plt.title('Training Samples per Produce Category', fontsize=14, fontweight='bold')
plt.xlabel('Category', fontsize=12)
plt.ylabel('Number of Images', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Produce Gallery (36 Categories Grid)
Visual inspection of sample images across all 36 categories.

In [ ]:
classes = sorted(list(train_counts.keys()))
rows, cols = 6, 6
fig, axes = plt.subplots(rows, cols, figsize=(18, 18))
axes = axes.flatten()

for idx, cls_name in enumerate(classes):
    cls_dir = train_dir / cls_name
    sample_img_path = next(cls_dir.glob('*.*'))
    img = Image.open(sample_img_path).convert('RGB')
    
    axes[idx].imshow(img)
    axes[idx].set_title(cls_name.title(), fontsize=10, fontweight='bold')
    axes[idx].axis('off')

plt.suptitle('36 Produce Categories — Sample Gallery', fontsize=18, fontweight='bold', y=0.92)
plt.tight_layout()
plt.show()

## 3. Image Dimensions & Resolution Distribution
Checking image dimensions across the training set to validate resize/crop settings.

In [ ]:
widths, heights, aspect_ratios = [], [], []

for cls_dir in train_dir.iterdir():
    if not cls_dir.is_dir():
        continue
    for img_path in list(cls_dir.glob('*.*'))[:10]:  # Sample 10 per class for speed
        with Image.open(img_path) as im:
            w, h = im.size
            widths.append(w)
            heights.append(h)
            aspect_ratios.append(w / h)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.scatter(widths, heights, alpha=0.4, color='#3498db', edgecolors='none')
ax1.set_title('Width vs. Height Distribution', fontsize=12, fontweight='bold')
ax1.set_xlabel('Width (px)')
ax1.set_ylabel('Height (px)')
ax1.grid(alpha=0.3)

ax2.hist(aspect_ratios, bins=30, color='#e67e22', edgecolor='#d35400')
ax2.set_title('Aspect Ratio Distribution', fontsize=12, fontweight='bold')
ax2.set_xlabel('Aspect Ratio (Width / Height)')
ax2.set_ylabel('Frequency')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Median Width : {np.median(widths):.0f} px")
print(f"Median Height: {np.median(heights):.0f} px")